<a href="https://colab.research.google.com/github/plasmaade34/liquid-telemetry-research/blob/main/SimScale_Run_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install git+https://github.com/SimScaleGmbH/simscale-python-sdk.git

  Cloning https://github.com/SimScaleGmbH/simscale-python-sdk.git to /tmp/pip-req-build-wvz0bqel
  Running command git clone --filter=blob:none --quiet https://github.com/SimScaleGmbH/simscale-python-sdk.git /tmp/pip-req-build-wvz0bqel
  Resolved https://github.com/SimScaleGmbH/simscale-python-sdk.git to commit 2989f25489ed77db3e8e1778c1dd24f4a84a4303
  Preparing metadata (setup.py) ... done
  Created wheel for simscale-sdk: filename=simscale_sdk-19.1.0-py3-none-any.whl size=3750004 sha256=9ff318792365d482d19e6ade33712882fd5197e1603bdaced25d909028140da2
  Stored in directory: /tmp/pip-ephem-wheel-cache-gj8qmsyl/wheels/6f/0e/8d/de402fab2cd3bcd337f144184c53ef9bf05548ebcb6c800fba
Successfully built simscale-sdk


In [ ]:
# 1. Install SimScale SDK
!pip install simscale-sdk -q

import simscale_sdk as sim

# 2. Configure API Client with your key
API_KEY = "d7de026b-e3b2-453d-ad4a-0f540995885c"

configuration = sim.Configuration()
configuration.host = "https://api.simscale.com/v1"
configuration.api_key['X-API-KEY'] = API_KEY

api_client = sim.ApiClient(configuration)

# 3. Instantiate APIs
projects_api = sim.ProjectsApi(api_client)
simulations_api = sim.SimulationsApi(api_client)

# 4. Create Project
project_payload = sim.Project(
    name="22oz_Hydration_Telemetry_Slosh",
    description="Multiphase dynamic slosh simulation for 22 oz container volume telemetry",
    measurement_system="SI"
)

created_project = projects_api.create_project(project=project_payload)
project_id = created_project.project_id

print("Project created successfully!")
print(f"Project Name : {created_project.name}")
print(f"Project ID   : {project_id}")

# 5. Define Multiphase Model Spec
multiphase_model = sim.Multiphase(
    type="MULTIPHASE",
    is_compressible=False,
    turbulence_model="K_EPSILON",
    model=sim.FluidModel(
        gravitational_acceleration=sim.DimensionalVectorAcceleration(
            value={'x': 0.0, 'y': 0.0, 'z': -9.81},
            unit="m/s²"
        )
    )
)

sim_spec = sim.SimulationSpec(
    name="22oz_Water_Slosh_Kinematic_Run",
    model=multiphase_model
)

print("\nMultiphase configuration ready.")
print(f"Next: Link CAD geometry to Project ID: {project_id}")

Project created successfully!
Project Name : 22oz_Hydration_Telemetry_Slosh
Project ID   : 5874714708388522555


TypeError: FluidModel.__init__() got an unexpected keyword argument 'gravitational_acceleration'

In [ ]:
import os
import time
import requests
import simscale_sdk as sim

# 1. API Client Configuration
API_KEY = "d7de026b-e3b2-453d-ad4a-0f540995885c"
configuration = sim.Configuration()
configuration.host = "https://api.simscale.com/v1"
configuration.api_key['X-API-KEY'] = API_KEY

api_client = sim.ApiClient(configuration)

# 2. Service Endpoints
storage_api = sim.StorageApi(api_client)
geometry_imports_api = sim.GeometryImportsApi(api_client)

# Your Project ID from the previous run
PROJECT_ID = "5874714708388522555"

# Local STEP file path in your Colab files
LOCAL_CAD_PATH = "vessel_22oz.step"  # Update filename if named differently

# --------------------------------------------------------------------------
# Step 1: Request Storage Allocation
# --------------------------------------------------------------------------
print(f"Requesting storage allocation for {LOCAL_CAD_PATH}...")
storage_response = storage_api.create_storage()
storage_id = storage_response.storage_id
upload_url = storage_response.url

# --------------------------------------------------------------------------
# Step 2: Upload Raw Binary
# --------------------------------------------------------------------------
print("Uploading CAD binary to SimScale storage...")
with open(LOCAL_CAD_PATH, "rb") as f:
    resp = requests.put(upload_url, data=f, headers={"Content-Type": "application/octet-stream"})

if resp.status_code not in (200, 201):
    raise RuntimeError(f"Binary upload failed. HTTP Status: {resp.status_code}")

print("Upload complete.")

# --------------------------------------------------------------------------
# Step 3: Trigger Geometry Import into Project
# --------------------------------------------------------------------------
print(f"Importing CAD into project {PROJECT_ID}...")
import_request = sim.GeometryImportRequest(
    name="22oz_Vessel_CAD",
    location=sim.GeometryImportRequestLocation(storage_id=storage_id),
    format="STEP"
)

import_job = geometry_imports_api.import_geometry(
    project_id=PROJECT_ID,
    geometry_import_request=import_request
)

import_id = import_job.geometry_import_id
print(f"Import task started: {import_id}. Parsing geometry...")

# --------------------------------------------------------------------------
# Step 4: Poll Until Ready
# --------------------------------------------------------------------------
while True:
    status_response = geometry_imports_api.get_geometry_import(
        project_id=PROJECT_ID,
        geometry_import_id=import_id
    )
    status = status_response.status

    if status == "FINISHED":
        geometry_id = status_response.geometry_id
        print(f"\nGeometry successfully imported!")
        print(f"Geometry ID: {geometry_id}")
        break
    elif status == "FAILED":
        raise RuntimeError(f"Geometry import failed: {status_response.failure_reason}")
    else:
        print(f"Status: {status}... checking in 5s")
        time.sleep(5)

Requesting storage allocation for vessel_22oz.step...
Uploading CAD binary to SimScale storage...


FileNotFoundError: [Errno 2] No such file or directory: 'vessel_22oz.step'

In [ ]:
# Install OpenCASCADE CAD engine and SimScale SDK
!pip install build123d simscale-sdk requests -q

import os
import time
import requests
from build123d import *
import simscale_sdk as sim

print("Environment libraries successfully installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.8/387.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.0/639.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 

In [ ]:
# Cell 2: Procedural 22 oz Internal Fluid Volume Generator
from build123d import *
import os

# Physical dimensions (mm)
RADIUS_MM = 37.0       # 74 mm internal diameter (cylindrical profile)
HEIGHT_MM = 174.5      # 174.5 mm internal height
FILLET_RADIUS_MM = 3.0 # Internal base radius

print("Building 22 oz fluid geometry core...")
with BuildPart() as fluid_domain:
    # 1. Primary cylindrical volume
    Cylinder(radius=RADIUS_MM, height=HEIGHT_MM, align=(Align.CENTER, Align.CENTER, Align.MIN))

    # 2. Add base fillet matching vessel bottom geometry
    bottom_edge = fluid_domain.edges().sort_by(Axis.Z)[0]
    fillet(bottom_edge, radius=FILLET_RADIUS_MM)

# Export directly to STEP format
CAD_OUTPUT_PATH = "/content/vessel_22oz.step"
export_step(fluid_domain.part, CAD_OUTPUT_PATH)

file_size_kb = os.path.getsize(CAD_OUTPUT_PATH) / 1024
total_vol_ml = fluid_domain.part.volume / 1000.0

print(f"\nSTEP file successfully generated: {CAD_OUTPUT_PATH}")
print(f"File Size: {file_size_kb:.1f} KB")
print(f"Calculated Internal Fluid Volume: {total_vol_ml:.2f} mL (~22 fl oz + headspace)")

Building 22 oz fluid geometry core...

STEP file successfully generated: /content/vessel_22oz.step
File Size: 8.0 KB
Calculated Internal Fluid Volume: 750.06 mL (~22 fl oz + headspace)


In [ ]:
# Cell 3: Create Project, Upload STEP, Parse CAD, and Extract Surface IDs
import os
import time
import requests
import simscale_sdk as sim

# 1. API Configuration
API_KEY = "d7de026b-e3b2-453d-ad4a-0f540995885c"
CAD_FILE_PATH = "/content/vessel_22oz.step"

if not os.path.exists(CAD_FILE_PATH):
    raise FileNotFoundError(f"Missing CAD file at {CAD_FILE_PATH}. Run Cell 2 first.")

configuration = sim.Configuration()
configuration.host = "https://api.simscale.com/v1"
configuration.api_key['X-API-KEY'] = API_KEY

api_client = sim.ApiClient(configuration)

# 2. Service Endpoints
projects_api = sim.ProjectsApi(api_client)
storage_api = sim.StorageApi(api_client)
geometry_imports_api = sim.GeometryImportsApi(api_client)
geometries_api = sim.GeometriesApi(api_client)

# 3. Create or Bind Project dynamically to prevent 404
print("Creating project dynamically to prevent 404 errors...")
project_payload = sim.Project(
    name="22oz_Hydration_Telemetry_Slosh",
    description="Multiphase dynamic slosh simulation for 22 oz container volume telemetry",
    measurement_system="SI"
)
created_project = projects_api.create_project(project=project_payload)
PROJECT_ID = created_project.project_id
print(f"Active Project Bound - Name: {created_project.name}, ID: {PROJECT_ID}")

# Pause for 5 seconds to ensure backend synchronization of the new project ID
print("Pausing 5 seconds for backend synchronization...")
time.sleep(5)

# 4. Request Storage Upload URL
print(f"Requesting storage allocation for: {CAD_FILE_PATH}...")
storage_response = storage_api.create_storage()
storage_id = storage_response.storage_id
upload_url = storage_response.url

# 5. Upload STEP File
print("Uploading STEP file binary...")
with open(CAD_FILE_PATH, "rb") as f:
    resp = requests.put(upload_url, data=f, headers={"Content-Type": "application/octet-stream"})

if resp.status_code not in (200, 201):
    raise RuntimeError(f"Binary upload failed. HTTP Status: {resp.status_code}")

print("Upload complete.")

# 6. Trigger Geometry Import
print(f"Importing CAD into project {PROJECT_ID}...")
import_request = sim.GeometryImportRequest(
    name="22oz_Vessel_CAD",
    location=sim.GeometryImportRequestLocation(storage_id=storage_id),
    format="STEP"
)

import_job = geometry_imports_api.import_geometry(
    project_id=PROJECT_ID,
    geometry_import_request=import_request
)

import_id = import_job.geometry_import_id
print(f"Import task started: {import_id}. Parsing geometry...")

# 7. Poll Until Ready
geometry_id = None
while True:
    status_response = geometry_imports_api.get_geometry_import(
        project_id=PROJECT_ID,
        geometry_import_id=import_id
    )
    status = status_response.status

    if status == "FINISHED":
        geometry_id = status_response.geometry_id
        print(f"\nGeometry successfully imported!")
        print(f"Geometry ID: {geometry_id}")
        break
    elif status == "FAILED":
        raise RuntimeError(f"Geometry import failed: {status_response.failure_reason}")
    else:
        print(f"Status: {status}... checking in 5s")
        time.sleep(5)

Creating project dynamically to prevent 404 errors...
Active Project Bound - Name: 22oz_Hydration_Telemetry_Slosh, ID: 9148140533139949593
Requesting storage allocation for: /content/vessel_22oz.step...
Uploading STEP file binary...
Upload complete.
Importing CAD into project 9148140533139949593...


ApiException: (404)
Reason: 
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json', 'Content-Length': '145', 'Connection': 'keep-alive', 'Date': 'Wed, 07 Oct 2026 18:45:30 GMT', 'Expires': 'Thu, 01 Jan 1970 00:00:00 UTC', 'Vary': 'Accept-Encoding, Origin, Access-Control-Request-Method, Access-Control-Request-Headers, Authorization, Origin', 'Trace-Id': '50cd704c, 50cd704c', 'Span-Id': 'p5yq64kp, 3buyspni', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'X-Frame-Options': 'DENY', 'Cache-Control': 'private', 'X-Cache': 'Error from cloudfront', 'Via': '1.1 afd508e8725b6457725f7b20366832e6.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'IAD61-P9', 'Alt-Svc': 'h3=":443"; ma=86400', 'X-Amz-Cf-Id': 'SWTpORZRuIX_7wRteHBxJrYaZJVb6JutlEsz1WDqnwpiMtfTfPm8yA==', 'Referrer-Policy': 'strict-origin-when-cross-origin'})
HTTP response body: {
  "code" : "not-found",
  "details" : null,
  "entr